# 라이브러리 설치

In [ ]:
%pip install torch torchvision pandas onnx onnxscript torchmetrics welford

Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py) ... done
  Created wheel for welford: filename=welford-0.2.5-py3-none-any.whl size=5492 sha256=cbab9d3381f89a1de12e8fce0fcfdb7e76b5718df662b4a7ddeb778f7886b68e
  Stored in directory: /home/an0jin/.cache/pip/wheels/86/f7/09/0d69ed91cd595e4eeebb41bcf6aef54cf587af25caf0b9e0dd
Successfully built welford
Note: you may need to restart the kernel to use updated packages.


In [35]:
from torchmetrics.classification import Accuracy, F1Score, Recall, Precision
from torchmetrics import MetricCollection
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
import pandas as pd
from model import Model
import os
from welford import Welford

In [36]:
BATCH = 64
EPOCHS = 300
LR = 0.01
IMGZ = (640,640)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [37]:
img_dir = './dataset/'
folderNames = ['train', 'val']
final_result = {}

for folderName in folderNames:
    current_path = os.path.join(img_dir, folderName)
    if not os.path.exists(current_path):
        continue

    dataset = datasets.ImageFolder(
        root=current_path,
        transform=transforms.Compose([
            transforms.Resize(IMGZ),
            transforms.ToTensor()
        ])
    )
    loader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=4)

    channel_sum = torch.zeros(3)
    channel_sum_sq = torch.zeros(3)
    pixel_count = 0

    for imgs, _ in loader:
        b, c, h, w = imgs.shape
        channel_sum += imgs.sum(dim=[0, 2, 3])
        channel_sum_sq += (imgs ** 2).sum(dim=[0, 2, 3])
        pixel_count += b * h * w

    mean = channel_sum / pixel_count
    std = torch.sqrt(channel_sum_sq / pixel_count - mean ** 2)

    final_result[folderName] = {
        "mean": mean.tolist(),
        "std": std.tolist()
    }

print(final_result)

{'train': {'mean': [0.6227040886878967, 0.5125784277915955, 0.4804546535015106], 'std': [0.2943863272666931, 0.26802992820739746, 0.26196956634521484]}, 'val': {'mean': [0.6864941120147705, 0.5493988394737244, 0.49836423993110657], 'std': [0.2562691271305084, 0.23706462979316711, 0.2357376515865326]}}


In [40]:
trainCopmpose=transforms.Compose([
        transforms.Resize(IMGZ),
    
    # fliplr: 0.5 (좌우 반전 50%)
    transforms.RandomHorizontalFlip(p=0.5),
    
    # flipud: 0.0 (상하 반전 안 함 - 생략 가능)
    # transforms.RandomVerticalFlip(p=0.0), 
    
    # auto_augment: randaugment (YOLO 내부 기본 RandAugment 적용)
    transforms.RandAugment(num_ops=2, magnitude=9),
    
    # hsv_h: 0.015, hsv_s: 0.7, hsv_v: 0.4 (Color Jitter로 유사 구현)
    transforms.ColorJitter(hue=0.015, saturation=0.7, brightness=0.4),
    
    # 텐서 변환 및 정규화 (YOLO는 기본 [0, 1] 스케일링 후 모델 내부 스케일링 수행)
    transforms.ToTensor(),
    
    # erasing: 0.4 (Random Erasing 40% 확률 적용)
    transforms.RandomErasing(p=0.4, scale=(0.02, 0.33), ratio=(0.3, 3.3), value=0),
    transforms.Normalize([0.6227040886878967, 0.5125784277915955, 0.4804546535015106],[0.2943863272666931, 0.26802992820739746, 0.26196956634521484])
    ])
val_Copmpose = transforms.Compose([
    # crop_fraction: 1.0 이므로 Resize만 수행
    transforms.Resize(IMGZ),
        transforms.ToTensor(),
    transforms.Normalize([0.6864941120147705, 0.5493988394737244, 0.49836423993110657], [0.2562691271305084, 0.23706462979316711, 0.2357376515865326]),
])

data_dir = './dataset' # 데이터셋 디렉토리 설정 (Set dataset directory)
# 데이터셋 불러오기 (Load datasets)
image_datasets={'train':datasets.ImageFolder(os.path.join(data_dir, 'train'),trainCopmpose),
                'val':datasets.ImageFolder(os.path.join(data_dir, 'val'),val_Copmpose)}
# 데이터 로더 설정 (Set up data loaders)
dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH, shuffle=True)
              for x in ['train', 'val']}

In [41]:
model=Model(len(image_datasets['train'].classes))
criterion = CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR)

In [42]:


# 평가 지표 통합 구성
base_metrics = MetricCollection({
    'Acc': Accuracy(task='multiclass', num_classes=model.num_classes),
    'F1': F1Score(task='multiclass', num_classes=model.num_classes, average='macro'),
    'Prec': Precision(task='multiclass', num_classes=model.num_classes, average='macro'),
    'Rec': Recall(task='multiclass', num_classes=model.num_classes, average='macro')
})

# 접두사를 통해 상태 분리 및 디바이스 동기화
metrics = {
    'train': base_metrics.clone(prefix='train_').to(DEVICE),
    'val': base_metrics.clone(prefix='val_').to(DEVICE)
}


In [43]:
i=0
while True:
    try:
        save_path=f'run/train{"" if i==0 else i}'
        os.makedirs(save_path)
        print(f"{save_path}에 저장되었습니다")
        break
    except Exception as e:
        i+=1

run/train에 저장되었습니다


In [44]:
def save_csv(history,save_path):
    if len(history) > 0:
        df = pd.DataFrame(history)
        df.index=df.index+1
        df.to_csv(f'{save_path}/result.csv',index_label="epoch")
        
    else:
        print("저장할 학습 결과가 없습니다.")

In [ ]:
best_val_acc = 0.0
best_val_loss = float('inf')

history = []

for epoch in range(EPOCHS):
    epoch_results = {}
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
        
    for phase in dataloaders.keys():
        model.train() if phase == 'train' else model.eval()
        running_loss = 0.0
        metrics[phase].reset()# 이번 에폭의 정확도 계산을 위해 torchmetrics 지표 초기화
        
        with torch.set_grad_enabled(phase == 'train'):#학습 모드 설정
            for inputs, labels in dataloaders[phase]:# 배치 사이즈 크기만큼 데이터를 쪼개서 가져옴
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)#input은 데이터 label은 정답
                optimizer.zero_grad()#오답 노트를 깨끗이 비운다 (0으로 초기화)
                outputs = model(inputs)#모델 실행
                loss = criterion(outputs, labels)#손실값 계산
                
                if phase == 'train':
                    loss.backward()#방금 틀린 점수를 오답 노트에 적는다
                    optimizer.step()#오답 노트를 보고 컴퓨터의 지식을 실제로 수정한다
                
                running_loss += loss.item() * inputs.size(0)#현재 손실을 더한다 계산 * inputs.size(0)[배치,3(채널),높이,너비)를 * BATCH대신 쓰는 이유는 마지막의 공부할 이미지수가 batch보다 작을수 있어서
                metrics[phase].update(outputs, labels)
        
        phase_loss = running_loss / len(dataloaders[phase].dataset)#손실구하기
        phase_metrics = {k: v.item() for k, v in metrics[phase].compute().items()}
        
        epoch_results.update(phase_metrics)
        epoch_results[f'{phase}_Loss'] = phase_loss
        
        metric_str = " | ".join([f"{k}: {v:.4f}" for k, v in phase_metrics.items()])
        print(f"[{phase.upper()}] Loss: {phase_loss:.4f} | {metric_str}")
        
    history.append(epoch_results)
    save_csv(history,save_path)
    
    current_val_acc = epoch_results['val_Acc']
    current_val_loss = epoch_results['val_Loss']
    print(f"{current_val_acc}>{best_val_acc}={current_val_acc>best_val_acc}")
    print(f"({current_val_acc} == {best_val_acc}) and ({current_val_loss} < {best_val_loss})={(current_val_acc == best_val_acc) and (current_val_loss < best_val_loss)}")
    
    if current_val_acc is not None and current_val_loss is not None:
        is_best_acc = current_val_acc > best_val_acc
        is_equal_acc_but_better_loss = (current_val_acc == best_val_acc) and (current_val_loss < best_val_loss)
        
        if is_best_acc or is_equal_acc_but_better_loss:
            best_val_acc = current_val_acc
            best_val_loss = current_val_loss
            
            best_path = os.path.join(save_path, "best.pt")
            torch.save(model.state_dict, best_path)
            print(f"--> [SAVED Best] Accuracy: {best_val_acc:.4f} | Loss: {best_val_loss:.4f}")

    last_path = os.path.join(save_path, "last.pt")
    torch.save(model.state_dict, last_path)
print(f"\n--> [SAVED Last] Final model saved at epoch {EPOCHS}.")


Epoch 1/300
[TRAIN] Loss: 4.6427 | train_Acc: 0.2941 | train_F1: 0.2925 | train_Prec: 0.2922 | train_Rec: 0.2929
[VAL] Loss: 3.2179 | val_Acc: 0.3974 | val_F1: 0.3595 | val_Prec: 0.3734 | val_Rec: 0.3899
0.39743590354919434>0.0=True
(0.39743590354919434 == 0.0) and (3.2179337532092362 < inf)=False
--> [SAVED Best] Accuracy: 0.3974 | Loss: 3.2179

Epoch 2/300
[TRAIN] Loss: 3.2143 | train_Acc: 0.3816 | train_F1: 0.3803 | train_Prec: 0.3811 | train_Rec: 0.3799
[VAL] Loss: 2.6473 | val_Acc: 0.4410 | val_F1: 0.4045 | val_Prec: 0.4154 | val_Rec: 0.4331
0.44102564454078674>0.39743590354919434=True
(0.44102564454078674 == 0.39743590354919434) and (2.647289491311098 < 3.2179337532092362)=False
--> [SAVED Best] Accuracy: 0.4410 | Loss: 2.6473

Epoch 3/300
[TRAIN] Loss: 2.5797 | train_Acc: 0.4004 | train_F1: 0.3977 | train_Prec: 0.3975 | train_Rec: 0.3983
[VAL] Loss: 2.2192 | val_Acc: 0.4590 | val_F1: 0.4399 | val_Prec: 0.4686 | val_Rec: 0.4605
0.45897436141967773>0.44102564454078674=True
(0.458